In [1]:
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
# Load dataset
weather_data = pd.read_csv("clean_weather.csv")
weather_data = weather_data.ffill()

print("Dataset shape:", weather_data.shape)
print("\nFirst 5 rows:")
weather_data.head()

Dataset shape: (13509, 5)

First 5 rows:


,Unnamed: 0,tmax,tmin,rain,tmax_tomorrow
0,1970-01-01,60.0,35.0,0.0,52.0
1,1970-01-02,52.0,39.0,0.0,52.0
2,1970-01-03,52.0,35.0,0.0,53.0
3,1970-01-04,53.0,36.0,0.0,52.0
4,1970-01-05,52.0,35.0,0.0,50.0


In [3]:
# Define input features and target variable
INPUT_FEATURES = ["tmax", "tmin", "rain"]
TARGET_COL = "tmax_tomorrow"

# Standardize features (mean=0, std=1)
scaler = StandardScaler()
weather_data[INPUT_FEATURES] = scaler.fit_transform(weather_data[INPUT_FEATURES])

print("Features after scaling:")
weather_data[INPUT_FEATURES].describe().round(3)

Features after scaling:


,tmax,tmin,rain
count,13509.000,13509.000,13509.000
mean,0.000,0.000,0.000
std,1.000,1.000,1.000
min,-3.369,-7.277,-0.254
25%,-0.727,-0.652,-0.254
50%,-0.007,0.084,-0.254
75%,0.593,0.820,-0.254
max,6.717,2.734,19.572


In [ ]:
# Split into train (70%), validation (15%), test (15%)
np.random.seed(67)
data_splits = np.split(
    weather_data,
    [int(0.70 * len(weather_data)), int(0.85 * len(weather_data))]
)

(train_x, train_y), (val_x, val_y), (test_x, test_y) = [
    [split[INPUT_FEATURES].to_numpy(), split[[TARGET_COL]].to_numpy()]
    for split in data_splits
]

print(f"Train:      {train_x.shape}")
print(f"Validation: {val_x.shape}")
print(f"Test:       {test_x.shape}")

Train:      (9456, 3)
Validation: (2026, 3)
Test:       (2027, 3)


c:\Users\akari\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
def mse_loss(actual, predicted):
    """Mean Squared Error loss"""
    return np.mean((predicted - actual) ** 2)

def mse_gradient(actual, predicted):
    """Gradient of MSE loss with respect to predictions"""
    return (predicted - actual) / actual.shape[0]

In [ ]:
def init_weights(layer_config):
    """
    Initialize weights for each RNN layer using Kaiming initialization.
    k = 1/sqrt(hidden_size) ensures weights are in tanh's stable region.
    """
    rnn_layers = []
    
    for i in range(1, len(layer_config)):
        np.random.seed(42)
        hidden_size = layer_config[i]["hidden"]
        k = 1.0 / math.sqrt(hidden_size)
        
        prev_units = layer_config[i-1]["units"]
        output_size = layer_config[i]["output"]
        
        # Input → Hidden weights
        w_input  = np.random.rand(prev_units,   hidden_size) * 2 * k - k
        # Hidden → Hidden weights (square matrix)
        w_hidden = np.random.rand(hidden_size,  hidden_size) * 2 * k - k
        # Hidden bias
        b_hidden = np.random.rand(1,            hidden_size) * 2 * k - k
        # Hidden → Output weights
        w_output = np.random.rand(hidden_size,  output_size) * 2 * k - k
        # Output bias
        b_output = np.random.rand(1,            output_size) * 2 * k - k
        
        rnn_layers.append([w_input, w_hidden, b_hidden, w_output, b_output])
    
    return rnn_layers

## 4. Forward Pass

In [ ]:
def forward_pass(x, rnn_layers):
    """
    Forward pass through all RNN layers.
    At each timestep: h_t = tanh(x_t @ W_i + h_{t-1} @ W_h + b_h)
    """
    all_hiddens = []
    all_outputs = []
    
    for layer_idx in range(len(rnn_layers)):
        w_input, w_hidden, b_hidden, w_output, b_output = rnn_layers[layer_idx]
        
        # Initialize hidden state and output arrays
        hidden_states = np.zeros((x.shape[0], w_input.shape[1]))
        layer_outputs = np.zeros((x.shape[0], w_output.shape[1]))
        
        for t in range(x.shape[0]):
            # Input contribution
            input_contrib  = x[t, :][np.newaxis, :] @ w_input
            
            # Hidden state: combine input + previous hidden
            h_raw = input_contrib + hidden_states[max(t-1, 0), :][np.newaxis, :] @ w_hidden + b_hidden
            
            # Apply tanh activation
            h_activated = np.tanh(h_raw)
            hidden_states[t, :] = h_activated.flatten()
            
            # Output layer (no activation for regression)
            out = h_activated @ w_output + b_output
            layer_outputs[t, :] = out.flatten()
        
        all_hiddens.append(hidden_states)
        all_outputs.append(layer_outputs)
    
    # Return all hidden states + final layer output
    return all_hiddens, all_outputs[-1]

## 5. Backward Pass (BPTT)

In [ ]:
def backward_pass(rnn_layers, x, learning_rate, loss_grad, all_hiddens):
    """
    Backpropagation Through Time (BPTT).
    Gradients flow backward through timesteps AND layers.
    """
    for layer_idx in range(len(rnn_layers)):
        w_input, w_hidden, b_hidden, w_output, b_output = rnn_layers[layer_idx]
        hidden_states = all_hiddens[layer_idx]
        
        # Initialize gradient accumulators
        g_w_input  = 0
        g_w_hidden = 0
        g_b_hidden = 0
        g_w_output = 0
        g_b_output = 0
        
        next_hidden_grad = None
        
        # Iterate backward through timesteps
        for t in range(x.shape[0] - 1, -1, -1):
            out_grad = loss_grad[t, :][np.newaxis, :]
            
            # Output weight gradients
            g_w_output += hidden_states[t, :][:, np.newaxis] @ out_grad
            g_b_output += out_grad
            
            # Propagate error back to hidden layer
            h_grad = out_grad @ w_output.T
            
            # Add gradient from next timestep if exists
            if t < x.shape[0] - 1:
                h_grad += next_hidden_grad @ w_hidden.T
            
            # Apply tanh derivative: d/dx tanh(x) = 1 - tanh²(x)
            tanh_grad = 1 - hidden_states[t, :][np.newaxis, :] ** 2
            h_grad = np.multiply(h_grad, tanh_grad)
            
            # Store for previous timestep
            next_hidden_grad = h_grad.copy()
            
            # Hidden weight gradients (only if not first timestep)
            if t > 0:
                g_w_hidden += hidden_states[t-1, :][:, np.newaxis] @ h_grad
                g_b_hidden += h_grad
            
            # Input weight gradients
            g_w_input += x[t, :][:, np.newaxis] @ h_grad
        
        # Normalize by sequence length
        scaled_lr = learning_rate / x.shape[0]
        
        # Update weights
        w_input  -= g_w_input  * scaled_lr
        w_hidden -= g_w_hidden * scaled_lr
        b_hidden -= g_b_hidden * scaled_lr
        w_output -= g_w_output * scaled_lr
        b_output -= g_b_output * scaled_lr
        
        rnn_layers[layer_idx] = [w_input, w_hidden, b_hidden, w_output, b_output]
    
    return rnn_layers

## 6. Training

In [ ]:
# Hyperparameters
NUM_EPOCHS   = 250
LEARNING_RATE = 1e-5
WINDOW_SIZE  = 7   # look back 7 days

# Model architecture
model_config = [
    {"type": "input", "units": 3},
    {"type": "rnn",   "hidden": 8, "output": 1}   # increased hidden size to 8
]

# Initialize weights
model_layers = init_weights(model_config)

# Track losses for plotting
train_losses = []
val_losses   = []
loss_epochs  = []

print("Starting training...\n")

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    
    # Training loop
    for j in range(train_x.shape[0] - WINDOW_SIZE):
        seq_x = train_x[j:(j + WINDOW_SIZE), ]
        seq_y = train_y[j:(j + WINDOW_SIZE), ]
        
        all_hiddens, predictions = forward_pass(seq_x, model_layers)
        grad = mse_gradient(seq_y, predictions)
        model_layers = backward_pass(model_layers, seq_x, LEARNING_RATE, grad, all_hiddens)
        epoch_loss += mse_loss(seq_y, predictions)
    
    # Validation every 50 epochs
    if epoch % 50 == 0:
        val_loss = 0
        
        for j in range(val_x.shape[0] - WINDOW_SIZE):
            seq_x = val_x[j:(j + WINDOW_SIZE), ]
            seq_y = val_y[j:(j + WINDOW_SIZE), ]
            _, predictions = forward_pass(seq_x, model_layers)
            val_loss += mse_loss(seq_y, predictions)
        
        avg_train = epoch_loss / len(train_x)
        avg_val   = val_loss   / len(val_x)
        
        train_losses.append(avg_train)
        val_losses.append(avg_val)
        loss_epochs.append(epoch)
        
        print(f"Epoch {epoch:>3} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

print("\nTraining complete!")

## 7. Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_epochs, train_losses, label="Train Loss", color="blue")
plt.plot(loss_epochs, val_losses,   label="Val Loss",   color="orange")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Evaluate on Test Set

In [ ]:
# Generate predictions on test set
all_predictions = []
all_actuals     = []

for j in range(test_x.shape[0] - WINDOW_SIZE):
    seq_x = test_x[j:(j + WINDOW_SIZE), ]
    seq_y = test_y[j:(j + WINDOW_SIZE), ]
    _, preds = forward_pass(seq_x, model_layers)
    all_predictions.append(preds[-1, 0])
    all_actuals.append(seq_y[-1, 0])

all_predictions = np.array(all_predictions)
all_actuals     = np.array(all_actuals)

test_mse = mse_loss(all_actuals, all_predictions)
print(f"Test MSE: {test_mse:.4f}")
print(f"Test RMSE: {np.sqrt(test_mse):.4f}")

## 9. Plot Predictions vs Actual

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(all_actuals[:100],     label="Actual",    color="blue",   linewidth=1.5)
plt.plot(all_predictions[:100], label="Predicted", color="red",    linewidth=1.5, linestyle="--")
plt.xlabel("Day")
plt.ylabel("Max Temperature")
plt.title("Predicted vs Actual Temperature (First 100 Test Days)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()